## B3.1 — AttentionMasker Demo

Self-attention occlusion via -inf injection.
Compares AttentionMasker vs VisionMeanMasker on dog & hydrant.

In [ ]:
import torch
torch.cuda.empty_cache()
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

import sys
sys.path.insert(0, '.')
from shapiq.imputer.vision import VisionImputerFactory
from shapiq.imputer.vision import MaskerConfig
from shapiq.imputer.vision import VisionLanguageGame

print('Imports OK')

In [ ]:
# Load TWO separate model instances with EAGER attention
model_mean = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32", attn_implementation="eager").to('cuda')
model_attn = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32", attn_implementation="eager").to('cuda')
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

input_text = "black dog next to a yellow hydrant"
input_image = Image.open("assets/dog_and_hydrant.png")

print('Models loaded (eager attention)')

In [ ]:
# Build VisionMean imputer (baseline)
factory = VisionImputerFactory()
imputer_mean = factory.build(model_mean, processor, input_image, input_text)
game_mean = VisionLanguageGame(imputer_mean, batch_size=64)

# Build AttentionMasker imputer
attn_cfg = MaskerConfig(strategy="attention")
imputer_attn = factory.build(model_attn, processor, input_image, input_text,
                              masker_config=attn_cfg)
game_attn = VisionLanguageGame(imputer_attn, batch_size=64)

print(f"Players: {game_attn.n_players} (img={game_attn.n_players_image}, txt={game_attn.n_players_text})")

In [ ]:
# Debug: verify hooks are registered
attn_masker = imputer_attn.masker
print(f"Hooks registered: {len(attn_masker._hooks)} (expected: 12)")
print(f"Patch size: {attn_masker._patch_size}, Grid: {attn_masker._grid_size}")
_ = game_attn.value_function(np.ones((1, game_attn.n_players), dtype=bool))
print(f"Hook call count after 1 fwd: {attn_masker._call_count} (expected: 12)")

In [ ]:
# Per-patch occlusion comparison + heatmap
n_players = game_attn.n_players
n_img = game_attn.n_players_image

co_full = np.ones((1, n_players), dtype=bool)
full_mean = game_mean.value_function(co_full)[0]
full_attn = game_attn.value_function(co_full)[0]
print(f"Full image: VisionMean={full_mean:.4f}  Attention={full_attn:.4f}")

drops_mean = np.zeros(n_img)
drops_attn = np.zeros(n_img)
print("Computing per-patch drop (~30s)...")
for p in range(n_img):
    co = np.ones((1, n_players), dtype=bool)
    co[0, p] = False
    drops_mean[p] = full_mean - game_mean.value_function(co)[0]
    drops_attn[p] = full_attn - game_attn.value_function(co)[0]
    if (p+1) % 7 == 0:
        print(f"  patch {p+1}/{n_img}  mean_drop={drops_mean[p-6:p+1].mean():.3f}  attn_drop={drops_attn[p-6:p+1].mean():.3f}")

print(f"\nVisionMean — min={drops_mean.min():.3f}  max={drops_mean.max():.3f}")
print(f"Attention  — min={drops_attn.min():.3f}  max={drops_attn.max():.3f}")

In [ ]:
# Plot: original image + VisionMean heatmap + Attention heatmap
img_array = np.array(input_image.resize((224, 224))) / 255.0

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_array)
axes[0].set_title('Original Image\n(dog & hydrant)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_array)
hm1 = axes[1].imshow(drops_mean.reshape(7, 7), cmap='Reds', alpha=0.7,
                      extent=[0, 224, 224, 0], vmin=0, vmax=max(drops_mean.max(), 0.01))
axes[1].set_title(f'VisionMeanMasker\n(max drop={drops_mean.max():.2f})', fontsize=12)
plt.colorbar(hm1, ax=axes[1], shrink=0.75)

axes[2].imshow(img_array)
hm2 = axes[2].imshow(drops_attn.reshape(7, 7), cmap='Reds', alpha=0.7,
                      extent=[0, 224, 224, 0], vmin=0, vmax=max(drops_attn.max(), drops_mean.max(), 0.01))
axes[2].set_title(f'AttentionMasker\n(max drop={drops_attn.max():.2f})', fontsize=12)
plt.colorbar(hm2, ax=axes[2], shrink=0.75)

fig.suptitle('B3.1 — Per-Patch Occlusion Heatmap (overlaid on image)', fontsize=14)
plt.tight_layout()
# plt.savefig('attention_masker_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved attention_masker_heatmap.png')

### Bonus: Run on 4 more MSCOCO images

In [ ]:
import datasets
ds = datasets.load_dataset("clip-benchmark/wds_mscoco_captions", split="test", streaming=True)
samples = []
for i, d in enumerate(ds):
    if i in [5, 12, 20, 35]:
        samples.append({"image": d['jpg'], "text": d['txt'].split('\n')[0], "id": i})
    if len(samples) >= 4:
        break
print(f"Loaded {len(samples)} extra images: {[s['id'] for s in samples]}")

In [ ]:
# Reusable function: compute per-patch heatmap for one image
def compute_heatmap(model_mean, model_attn, processor, image, text):
    factory = VisionImputerFactory()
    im_mean = factory.build(model_mean, processor, image, text)
    im_attn = factory.build(model_attn, processor, image, text,
                            masker_config=MaskerConfig(strategy="attention"))
    g_mean = VisionLanguageGame(im_mean, batch_size=64)
    g_attn = VisionLanguageGame(im_attn, batch_size=64)
    n_img = g_attn.n_players_image
    n_players = g_attn.n_players

    co_full = np.ones((1, n_players), dtype=bool)
    full_mean = g_mean.value_function(co_full)[0]
    full_attn = g_attn.value_function(co_full)[0]

    drops_mean = np.zeros(n_img)
    drops_attn = np.zeros(n_img)
    for p in range(n_img):
        co = np.ones((1, n_players), dtype=bool); co[0, p] = False
        drops_mean[p] = full_mean - g_mean.value_function(co)[0]
        drops_attn[p] = full_attn - g_attn.value_function(co)[0]
    return {"image": image, "text": text, "full": full_attn,
            "drops_mean": drops_mean, "drops_attn": drops_attn}

print("Function ready.")

In [ ]:
# Run heatmap on all 4 extra images (~2 min)
print("Running heatmap on 4 extra images...")
extra_results = []
for i, s in enumerate(samples):
    print(f"  [{i+1}/4] id={s['id']}: {s['text'][:50]}...", flush=True)
    extra_results.append(compute_heatmap(model_mean, model_attn, processor, s['image'], s['text']))
    torch.cuda.empty_cache()
print("Done.")

In [ ]:
# Plot: dog + 4 MSCOCO images (Attention heatmap only, 2x3 grid)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Dog
ax = axes[0]; img_arr = np.array(input_image.resize((224, 224))) / 255.0
ax.imshow(img_arr)
ax.imshow(drops_attn.reshape(7, 7), cmap='Reds', alpha=0.7,
          extent=[0, 224, 224, 0], vmin=0, vmax=max(drops_attn.max(), 0.01))
ax.set_title(f'#0 Dog & Hydrant\nmax drop={drops_attn.max():.2f}', fontsize=9)
ax.axis('off')

# 4 MSCOCO
for i, res in enumerate(extra_results):
    ax = axes[i+1]; img_arr = np.array(res['image'].resize((224, 224))) / 255.0
    ax.imshow(img_arr)
    ax.imshow(res['drops_attn'].reshape(7, 7), cmap='Reds', alpha=0.7,
              extent=[0, 224, 224, 0], vmin=0, vmax=max(res['drops_attn'].max(), 0.01))
    ax.set_title(f'#{samples[i]["id"]}: {res["text"][:35]}...\nmax drop={res["drops_attn"].max():.2f}', fontsize=8)
    ax.axis('off')

axes[5].set_visible(False)
fig.suptitle('B3.1 — AttentionMasker Heatmaps (5 images)', fontsize=14)
plt.tight_layout()
# plt.savefig('attention_masker_5images.png', dpi=150, bbox_inches='tight')
plt.show()